
# Topic 6 — Scaling, Evaluation & Production Fine-Tuning

**Core question:** *How do I make training reliable and ready for production?*

This is where you move from "I can fine-tune a model" to "I can run
fine-tuning as a production AI engineering practice." We build the pieces
around the LoRA model from Topic 2: an **evaluation harness**, a **model
registry**, **latency/throughput measurement**, and the **canary rollout**
workflow — plus the distributed-training ladder for when one GPU isn't enough.

```mermaid
flowchart LR
    A[Train] --> B[Evaluate]
    B --> C[Model registry]
    C --> D[Staging]
    D --> E[Shadow traffic]
    E --> F[Canary: small % of real traffic]
    F --> G[Production: 100%]
    B -.fails threshold.-> X[Reject / iterate]
```

```mermaid
flowchart LR
    S1[1 GPU] --> S2[Data Parallel]
    S2 --> S3[FSDP]
    S3 --> S4[Tensor Parallel]
    S4 --> S5[Pipeline Parallel]
```

### Can you answer this?
1. How do you evaluate whether fine-tuning actually helped?
2. How do you scale fine-tuning beyond one GPU?
3. How do you safely take a trained model to production?


In [1]:

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

console = Console()
def banner(title, subtitle="", style="bold white on blue"):
    text = title if not subtitle else f"{title}\n[dim]{subtitle}[/dim]"
    console.print(Panel(text, style=style, expand=False, box=box.DOUBLE))
def section(title):
    console.print(f"\n[bold cyan]── {title} " + "─" * max(0, 60 - len(title)) + "[/bold cyan]")
def kv_table(title, rows, caption=""):
    table = Table(title=title, box=box.SIMPLE_HEAVY, caption=caption)
    table.add_column("Key", style="bold")
    table.add_column("Value")
    for k, v in rows.items():
        table.add_row(str(k), str(v))
    console.print(table)

banner("Topic 6: Scaling, Evaluation & Production Fine-Tuning",
       "Training -> Evaluation -> Registry -> Serving -> Canary -> Production")


╔═══════════════════════════════════════════════════════════════════════╗
║ Topic 6: Scaling, Evaluation & Production Fine-Tuning                 ║
║ Training -> Evaluation -> Registry -> Serving -> Canary -> Production ║
╚═══════════════════════════════════════════════════════════════════════╝


## Concept primer

**Distributed training ladder** (as a model/dataset outgrow one GPU):
- **1 GPU**: everything fits -- what we've done so far.
- **Data Parallel (DP)**: the same full model is copied onto each GPU; each
  processes a different data shard and gradients are averaged/synced.
  Simple, but every GPU needs to hold the *entire* model.
- **FSDP (Fully Sharded Data Parallel)**: shards the model's parameters,
  gradients, and optimizer state *across* GPUs, gathering only what's
  needed for each computation -- lets you train models too big to fit on
  one GPU even with DP.
- **Tensor Parallel (TP)**: splits individual weight matrices *within* a
  layer across GPUs, so a single layer's computation itself is distributed.
- **Pipeline Parallel (PP)**: splits the model's *layers* across GPUs, so
  different GPUs hold different stages of the forward/backward pass,
  pipelined like an assembly line.

Real large-scale training often combines several of these (e.g. Megatron-LM
style TP + PP + DP) -- the right combination depends on model size, GPU
count, and interconnect bandwidth.

**Evaluation categories** -- fine-tuning without evaluation is just hoping:
- **Training/validation loss**: the most basic signal, but doesn't tell you
  about actual task quality.
- **Task accuracy**: does the model get the *right* answer on held-out
  examples (exact match, keyword overlap, or a verifier -- like Topic 4).
- **Instruction following**: does it do what was asked, in the format asked?
- **Human preference / LLM judge**: for open-ended quality, since there's
  often no single "correct" answer.
- **Regression tests**: a fixed suite of known-important cases that must
  keep working across every retrain.
- **Safety tests**: checks for harmful, biased, or policy-violating outputs.

**Production path**: `checkpoint -> model registry -> staging -> shadow ->
canary -> production`. Each stage reduces risk before the next: staging
validates the artifact works at all; shadow traffic runs the new model
alongside the old one on real traffic *without* serving its output to
users, just to compare; canary serves it to a small real percentage of
users; only after that holds up does it reach 100% production traffic.


In [2]:

import torch
section("Environment check")
HAS_CUDA = torch.cuda.is_available()
kv_table("Runtime", {"CUDA available": HAS_CUDA, "PyTorch": torch.__version__})
if not HAS_CUDA:
    console.print(Panel(
        "[bold yellow]No CUDA GPU detected.[/bold yellow] The evaluation harness, model registry, "
        "latency-percentile computation, and canary-rollout simulation below are all real, "
        "runnable CPU-safe work (using deterministic stand-in baselines instead of a real trained "
        "model). Loading the actual Topic 2 LoRA model and real vLLM serving require a GPU.",
        title="GPU NOT AVAILABLE HERE", style="yellow", box=box.HEAVY,
    ))


── Environment check ───────────────────────────────────────────

          Runtime          
                           
  Key              Value   
 ━━━━━━━━━━━━━━━━━━━━━━━━━ 
  CUDA available   False   
  PyTorch          2.14.1

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ GPU NOT AVAILABLE HERE ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ No CUDA GPU detected. The evaluation harness, model registry, latency-percentile computation, and               ┃
┃ canary-rollout simulation below are all real, runnable CPU-safe work (using deterministic stand-in baselines    ┃
┃ instead of a real trained model). Loading the actual Topic 2 LoRA model and real vLLM serving require a GPU.    ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

In [3]:

import sys
IN_COLAB = "google.colab" in sys.modules
packages = ["unsloth", "transformers", "trl", "peft", "bitsandbytes", "torch", "datasets", "rich", "vllm"]
if IN_COLAB:
    get_ipython().system(f"pip install -q {' '.join(packages[:-1])}")  # vllm installed separately/optionally
    console.print("[green]Installed packages for Colab (vllm optional -- see Step: serving).[/green]")
else:
    console.print("[dim]Not running in Colab -- assuming the project virtualenv already has these.[/dim]")


Not running in Colab -- assuming the project virtualenv already has these.

In [4]:
# --- Load the shared dataset (Topic 1/2's customer-support data) -----------
import json
import random
from pathlib import Path

section("Loading the shared dataset")

vsc_ipynb_file = globals().get("__vsc_ipynb_file__")
search_roots = [Path.cwd()]
if vsc_ipynb_file:
    search_roots.insert(0, Path(vsc_ipynb_file).resolve().parent)
search_roots += [root.parent for root in list(search_roots)]

RELATIVE_CANDIDATES = ["data/support_tickets.json", "06_scaling_eval_production/data/support_tickets.json",
                        "01_sft_fundamentals/data/support_tickets.json"]
DATA_PATH = None
for root in search_roots:
    for rel in RELATIVE_CANDIDATES:
        candidate = root / rel
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH:
        break

if DATA_PATH is None:
    # Self-sufficient fallback: regenerate the exact same deterministic dataset
    # in-memory instead of requiring a separate file upload (e.g. on Colab).
    console.print("[dim]support_tickets.json not found nearby -- generating it in-memory "
                  "instead of requiring a separate file upload.[/dim]")

    PRODUCTS = ["wireless earbuds", "coffee maker", "running shoes", "laptop stand",
                "desk lamp", "yoga mat", "bluetooth speaker", "phone case",
                "winter jacket", "kitchen blender", "backpack", "office chair",
                "water bottle", "gaming mouse", "electric toothbrush", "air fryer",
                "board game", "children's book", "garden hose", "car phone mount"]
    NAMES = ["Alex", "Jordan", "Priya", "Sam", "Taylor", "Morgan", "Chris",
             "Jamie", "Riley", "Casey", "Drew", "Avery"]

    def _order(): return f"#{random.randint(100000, 999999)}"
    def _days(lo=1, hi=14): return random.randint(lo, hi)

    def order_status():
        p, o, d = random.choice(PRODUCTS), _order(), _days(1, 10)
        return (f"I ordered a {p} with order number {o} {d} days ago and I still don't see any "
                f"shipping update. Can you tell me what's going on?",
                f"Thanks for reaching out about order {o}. I checked and your {p} is currently being "
                f"prepared at our fulfillment center and is on track to ship within 1-2 business days. "
                f"You'll receive a tracking email the moment it ships, and estimated delivery is "
                f"{_days(2,5)} days after that. I'll flag the order so it's monitored closely.")

    def returns_refunds():
        p, o = random.choice(PRODUCTS), _order()
        return (f"I want to return the {p} from order {o} because it's not what I expected. How do I "
                f"get a refund?",
                f"No problem at all -- I've started a return for the {p} in order {o}. You'll get a "
                f"prepaid return label by email within the hour. Once the item is scanned by the "
                f"carrier, your refund of the full purchase price will be issued to your original "
                f"payment method within 3-5 business days.")

    def damaged_item():
        p, o = random.choice(PRODUCTS), _order()
        return (f"The {p} I received in order {o} arrived damaged -- it looks like it was crushed in "
                f"shipping. What can you do?",
                f"I'm sorry the {p} arrived damaged, that's not the experience we want for you. I've "
                f"issued a free replacement for order {o} at no extra cost, and you don't need to "
                f"return the damaged item. The replacement will ship within 1 business day with "
                f"expedited shipping.")

    def payment_issue():
        o = _order()
        amount = random.choice([19.99, 29.99, 49.50, 89.00, 124.75])
        return (f"I was charged ${amount} twice for order {o}. Can you check my payment and fix this?",
                f"I checked order {o} and confirmed a duplicate charge of ${amount}. I've submitted a "
                f"refund for the extra charge, which will appear on your statement within 3-5 business "
                f"days. Only one charge of ${amount} will remain for your order.")

    def account_access():
        name = random.choice(NAMES)
        return (f"Hi, this is {name}. I can't log into my account, it keeps saying my password is "
                f"incorrect even after I reset it. Can you help?",
                f"Hi {name}, sorry for the trouble logging in. I've sent a fresh password reset link "
                f"to your account email -- it will be valid for 30 minutes. If it still doesn't work "
                f"after that, it may be a browser cache issue, so try an incognito window or clearing "
                f"cookies for our site.")

    def shipping_delay():
        p, o, d = random.choice(PRODUCTS), _order(), _days(5, 14)
        return (f"My {p} from order {o} was supposed to arrive {d} days ago and it still hasn't shown "
                f"up. This is really frustrating.",
                f"I completely understand the frustration, and I'm sorry for the delay on your {p} in "
                f"order {o}. I checked with the carrier and the package is delayed in transit but "
                f"still moving -- updated delivery is within 2 days. As an apology for the delay, I've "
                f"credited your account with a $10 voucher for your next order.")

    def cancellation():
        p, o = random.choice(PRODUCTS), _order()
        return (f"I need to cancel my order {o} for the {p} -- I accidentally ordered two of them. Is "
                f"it too late?",
                f"Good news -- order {o} hasn't shipped yet, so I was able to cancel it for you right "
                f"away. You will not be charged for this order, and any pending authorization will "
                f"drop off within a few business days. The other order for the {p} is unaffected.")

    def product_question():
        p = random.choice(PRODUCTS)
        return (f"Before I buy the {p}, can you tell me if it comes with a warranty and what the "
                f"return window is?",
                f"Great question! The {p} comes with a 1-year manufacturer warranty covering defects, "
                f"and you can return it within 30 days of delivery for any reason as long as it's in "
                f"like-new condition. If anything goes wrong after the return window, our support "
                f"team can still help with the manufacturer warranty.")

    def subscription_issue():
        return ("I signed up for a monthly subscription but I was charged again this month even "
                "though I thought I canceled it last week. Can you check?",
                "I looked into your subscription and I can confirm the cancellation request from "
                "last week did not fully process due to a billing sync issue on our end. I've now "
                "canceled the subscription effective immediately and refunded this month's charge "
                "in full -- it will reflect within 3-5 business days.")

    def promo_discount():
        code = random.choice(["SAVE10", "WELCOME15", "SUMMER20", "FIRST5"])
        return (f"I tried to use the promo code {code} at checkout but it says invalid. The order "
                f"total didn't get discounted. Can you help?",
                f"Thanks for flagging this. I checked the promo code {code} and applied the discount "
                f"manually to your most recent order, so you'll see the adjusted total reflected "
                f"shortly. In the future, that code works best when applied before adding a gift wrap "
                f"option, which was conflicting with it at checkout.")

    CATEGORIES = {
        "order_status": order_status, "returns_refunds": returns_refunds,
        "damaged_item": damaged_item, "payment_issue": payment_issue,
        "account_access": account_access, "shipping_delay": shipping_delay,
        "cancellation": cancellation, "product_question": product_question,
        "subscription_issue": subscription_issue, "promo_discount": promo_discount,
    }

    def generate_support_tickets(n=500, seed=42):
        rng_state = random.getstate()
        random.seed(seed)
        names = list(CATEGORIES.keys())
        per, rem = n // len(names), n % len(names)
        examples = []
        for i, name in enumerate(names):
            count = per + (1 if i < rem else 0)
            for _ in range(count):
                problem, response = CATEGORIES[name]()
                examples.append({"category": name, "problem": problem, "response": response})
        random.shuffle(examples)
        random.setstate(rng_state)
        return examples

    examples = generate_support_tickets()
    DATA_PATH = Path.cwd() / "data" / "support_tickets.json"
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    with DATA_PATH.open("w") as f:
        json.dump(examples, f, indent=2)
else:
    with DATA_PATH.open() as f:
        examples = json.load(f)

random.seed(7)
shuffled = examples[:]
random.shuffle(shuffled)
eval_set = shuffled[:30]
console.print(f"Loaded [bold]{len(examples)}[/bold] examples, using a [bold]{len(eval_set)}[/bold]-example held-out eval set")


── Loading the shared dataset ──────────────────────────────────

Loaded 500 examples, using a 30-example held-out eval set


## Building a real evaluation harness

To demonstrate the harness end-to-end without a GPU, we evaluate two
**deterministic stand-in baselines** instead of a real model: a "generic"
baseline (always replies with one canned sentence) and a "cheating"
baseline (echoes back the expected response verbatim, standing in for
"what a well-fine-tuned model should approximately achieve"). The harness
itself -- scoring, thresholds, pass/fail -- is exactly what you'd point at
real model outputs on Colab.


In [5]:

# --- Real, runnable evaluation harness (CPU-safe) ---------------------------
def keyword_overlap_score(generated, expected, k=6):
    expected_words = [w.strip(".,!?$#").lower() for w in expected.split() if len(w) > 4]
    key_terms = list(dict.fromkeys(expected_words))[:k]
    if not key_terms:
        return 0.0
    hits = sum(1 for term in key_terms if term in generated.lower())
    return hits / len(key_terms)

def generic_baseline(example):
    return "Thank you for reaching out, we will look into this and follow up shortly."

def cheating_baseline(example):
    return example["response"]  # stands in for "a well-trained model"

def evaluate(model_fn, eval_set, name):
    scores = [keyword_overlap_score(model_fn(ex), ex["response"]) for ex in eval_set]
    avg = sum(scores) / len(scores)
    return {"name": name, "avg_keyword_overlap": avg, "n": len(scores)}

section("Running the evaluation harness on two deterministic baselines")
results = [
    evaluate(generic_baseline, eval_set, "generic baseline (stand-in for an un-tuned model)"),
    evaluate(cheating_baseline, eval_set, "cheating baseline (stand-in for a well-fine-tuned model)"),
]

PASS_THRESHOLD = 0.6
table = Table(title="Evaluation harness results (real computation, stand-in models)", box=box.SIMPLE_HEAVY)
table.add_column("Candidate")
table.add_column("Avg. keyword overlap", justify="right")
table.add_column("Pass threshold (0.6)?")
for r in results:
    passed = "[green]PASS[/green]" if r["avg_keyword_overlap"] >= PASS_THRESHOLD else "[red]FAIL[/red]"
    table.add_row(r["name"], f"{r['avg_keyword_overlap']:.3f}", passed)
console.print(table)
console.print("\n[dim]On Colab, replace generic_baseline/cheating_baseline with calls to your actual "
              "base model and fine-tuned model to get a real quality signal this same way.[/dim]")


── Running the evaluation harness on two deterministic baselines 

                      Evaluation harness results (real computation, stand-in models)                       
                                                                                                           
  Candidate                                                  Avg. keyword overlap   Pass threshold (0.6)?  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  generic baseline (stand-in for an un-tuned model)                         0.050   FAIL                   
  cheating baseline (stand-in for a well-fine-tuned model)                  1.000   PASS

On Colab, replace generic_baseline/cheating_baseline with calls to your actual base model and fine-tuned model to 
get a real quality signal this same way.


## A toy model registry

A model registry tracks *which* trained artifact is where in the pipeline,
with its evaluation metrics attached -- so "which model is in production"
is never a guess. We implement a minimal file-based version to show the
real shape of the data, even without a real training run to register.


In [6]:

# --- Toy model registry (real file I/O, CPU-safe) ---------------------------
import time

REGISTRY_PATH = Path("model_registry.json")

def load_registry():
    if REGISTRY_PATH.is_file():
        with open(REGISTRY_PATH) as f:
            return json.load(f)
    return []

def register_model(version, metrics, stage="staging"):
    registry = load_registry()
    registry.append({
        "version": version,
        "metrics": metrics,
        "stage": stage,
        "registered_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    })
    with open(REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=2)
    return registry

section("Registering candidate models")
register_model("v1-lora-full_ft", {"avg_keyword_overlap": results[0]["avg_keyword_overlap"]}, stage="rejected")
registry = register_model("v2-lora-qlora", {"avg_keyword_overlap": results[1]["avg_keyword_overlap"]}, stage="staging")

table = Table(title=f"Model registry ({REGISTRY_PATH})", box=box.SIMPLE_HEAVY)
table.add_column("Version")
table.add_column("Stage")
table.add_column("avg_keyword_overlap")
table.add_column("Registered at")
for entry in registry:
    table.add_row(entry["version"], entry["stage"], f"{entry['metrics']['avg_keyword_overlap']:.3f}", entry["registered_at"])
console.print(table)


── Registering candidate models ────────────────────────────────

                   Model registry (model_registry.json)                   
                                                                          
  Version           Stage      avg_keyword_overlap   Registered at        
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  v1-lora-full_ft   rejected   0.050                 2026-10-02 18:16:52  
  v2-lora-qlora     staging    1.000                 2026-10-02 18:16:52  
  v1-lora-full_ft   rejected   0.050                 2026-10-02 20:21:28  
  v2-lora-qlora     staging    1.000                 2026-10-02 20:21:28  
  v1-lora-full_ft   rejected   0.050                 2026-10-02 20:43:45  
  v2-lora-qlora     staging    1.000                 2026-10-02 20:43:45

In [7]:

# --- Loading the real Topic 2 LoRA model for evaluation (GPU-only) ---------
if HAS_CUDA:
    from unsloth import FastLanguageModel
    model, tok = FastLanguageModel.from_pretrained(model_name="Qwen/Qwen3-0.6B", max_seq_length=512)
    console.print("[bold green]✓ Loaded base model -- attach your trained Topic 2 LoRA adapter here "
                  "and re-run the evaluation harness above against its real generations.[/bold green]")
else:
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- loading the real model to evaluate requires a CUDA GPU.\n"
        "Expected on Colab: load Qwen3-0.6B + the LoRA adapter trained in Topic 2, replace the\n"
        "stand-in baseline functions above with real `model.generate()` calls, and re-run the\n"
        "same evaluation harness and registry logic against real outputs.",
        title="Load real model for evaluation (GPU-only)", style="yellow",
    ))


╭─────────────────────────────────── Load real model for evaluation (GPU-only) ───────────────────────────────────╮
│ SKIPPED -- loading the real model to evaluate requires a CUDA GPU.                                              │
│ Expected on Colab: load Qwen3-0.6B + the LoRA adapter trained in Topic 2, replace the                           │
│ stand-in baseline functions above with real `model.generate()` calls, and re-run the                            │
│ same evaluation harness and registry logic against real outputs.                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Serving latency & throughput

**TTFT** (Time To First Token) and **TPOT** (Time Per Output Token) are the
two numbers that matter most for perceived chat latency. We compute real
p50/p95/p99 percentiles and tokens/sec from a latency distribution --
clearly labeled as **simulated** here (no GPU to measure from), but this is
exactly the computation you'd run on real measured request logs.


In [8]:

# --- Latency percentile computation (real math on simulated data) ----------
import numpy as np

section("Computing latency percentiles (simulated request latencies -- real math)")

rng = np.random.default_rng(7)
N_REQUESTS = 500
# Simulated, clearly illustrative distributions -- NOT measured on real hardware.
ttft_ms = rng.gamma(shape=4.0, scale=15.0, size=N_REQUESTS)       # time to first token
tpot_ms = rng.gamma(shape=6.0, scale=3.0, size=N_REQUESTS)        # time per subsequent token

p50, p95, p99 = np.percentile(ttft_ms, [50, 95, 99])
tp50, tp95, tp99 = np.percentile(tpot_ms, [50, 95, 99])
tokens_per_sec = 1000.0 / np.mean(tpot_ms)

table = Table(title="Latency percentiles (SIMULATED data -- real percentile math)", box=box.SIMPLE_HEAVY)
table.add_column("Metric")
table.add_column("p50", justify="right")
table.add_column("p95", justify="right")
table.add_column("p99", justify="right")
table.add_row("TTFT (ms)", f"{p50:.1f}", f"{p95:.1f}", f"{p99:.1f}")
table.add_row("TPOT (ms)", f"{tp50:.1f}", f"{tp95:.1f}", f"{tp99:.1f}")
console.print(table)
kv_table("Derived throughput", {"tokens/sec (1 / mean TPOT)": f"{tokens_per_sec:.1f}"})
console.print("\n[dim]On real hardware, collect actual per-request TTFT/TPOT from your serving "
              "layer (e.g. vLLM's metrics endpoint) and feed them into this exact percentile "
              "computation instead of the simulated arrays above.[/dim]")


── Computing latency percentiles (simulated request latencies -- real math) 

Latency percentiles (SIMULATED data 
      -- real percentile math)      
                                    
  Metric       p50     p95     p99  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  TTFT (ms)   52.1   109.9   148.0  
  TPOT (ms)   16.3    32.0    41.1

          Derived throughput          
                                      
  Key                          Value  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  tokens/sec (1 / mean TPOT)   56.7

On real hardware, collect actual per-request TTFT/TPOT from your serving layer (e.g. vLLM's metrics endpoint) and 
feed them into this exact percentile computation instead of the simulated arrays above.


## Serving with vLLM (GPU-only)

vLLM is a high-throughput inference server for LLMs (continuous batching,
PagedAttention for efficient KV-cache memory). Serving a fine-tuned model
(optionally with a LoRA adapter attached) through it is how you'd expose it
as a production API.


In [9]:

if HAS_CUDA:
    console.print(Panel(
        "To serve the Topic 2 LoRA model with vLLM on a GPU machine:\n\n"
        "  [bold]vllm serve Qwen/Qwen3-0.6B --enable-lora "
        "--lora-modules support-lora=outputs/lora_adapter[/bold]\n\n"
        "Then call it like any OpenAI-compatible API:\n\n"
        "  [bold]curl http://localhost:8000/v1/chat/completions \\\n"
        "    -d '{\"model\": \"support-lora\", \"messages\": [...]}'[/bold]\n\n"
        "vLLM also supports serving multiple LoRA adapters on one base model deployment "
        "(see Topic 2's scale discussion) -- this is how you'd serve many task/customer-specific "
        "adapters without duplicating the base model per adapter.",
        title="vLLM serving (reference -- GPU-only, not executed here)", style="cyan",
    ))
else:
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- vLLM serving requires a CUDA GPU and isn't exercised in this notebook.\n"
        "The reference commands above show the shape of a real deployment.",
        title="vLLM serving (GPU-only)", style="yellow",
    ))


╭──────────────────────────────────────────── vLLM serving (GPU-only) ────────────────────────────────────────────╮
│ SKIPPED -- vLLM serving requires a CUDA GPU and isn't exercised in this notebook.                               │
│ The reference commands above show the shape of a real deployment.                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Canary rollout simulation

A simple, runnable (CPU-safe) simulation of the staged rollout decision
logic: at each stage, check the candidate's eval metric against the
threshold before advancing traffic percentage -- and roll back immediately
if it ever fails.


In [10]:

# --- Canary rollout state machine (real, runnable simulation) ---------------
section("Simulating a staged canary rollout")

ROLLOUT_STAGES = [("staging", 0), ("shadow", 0), ("canary_5pct", 5), ("canary_25pct", 25), ("production", 100)]

def run_rollout(candidate_metric, threshold=0.6):
    log = []
    for stage_name, traffic_pct in ROLLOUT_STAGES:
        if candidate_metric < threshold:
            log.append((stage_name, traffic_pct, "ROLLED BACK -- metric below threshold"))
            return log
        log.append((stage_name, traffic_pct, "advanced"))
    return log

for name, metric in [("v2-lora-qlora (passed eval)", results[1]["avg_keyword_overlap"]),
                      ("v1-full_ft (failed eval)", results[0]["avg_keyword_overlap"])]:
    table = Table(title=f"Rollout simulation: {name}", box=box.SIMPLE_HEAVY)
    table.add_column("Stage")
    table.add_column("Traffic %", justify="right")
    table.add_column("Outcome")
    for stage_name, pct, outcome in run_rollout(metric, threshold=PASS_THRESHOLD):
        style = "green" if outcome == "advanced" else "red"
        table.add_row(stage_name, f"{pct}%", f"[{style}]{outcome}[/{style}]")
    console.print(table)


── Simulating a staged canary rollout ──────────────────────────

   Rollout simulation: v2-lora-qlora   
             (passed eval)             
                                       
  Stage          Traffic %   Outcome   
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  staging               0%   advanced  
  shadow                0%   advanced  
  canary_5pct           5%   advanced  
  canary_25pct         25%   advanced  
  production          100%   advanced

         Rollout simulation: v1-full_ft (failed eval)          
                                                               
  Stage     Traffic %   Outcome                                
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  staging          0%   ROLLED BACK -- metric below threshold


## Can you answer this? (Topic 6 training checkpoints)

**1. How do you evaluate whether fine-tuning actually helped?**
<details><summary>Answer</summary>
Compare the fine-tuned model against a clear baseline (the base model, or
the previous production version) on a held-out set using a defined metric
-- not just eyeballing a few outputs. Use multiple evaluation categories
(task accuracy, instruction following, regression tests, safety) since
each catches different failure modes; a single aggregate score can hide a
regression in one specific area.
</details>

**2. How do you scale fine-tuning beyond one GPU?**
<details><summary>Answer</summary>
Move up the distributed-training ladder based on what's actually
constrained: Data Parallel if the model fits on one GPU but you want more
throughput; FSDP if the model itself (plus optimizer state) doesn't fit on
one GPU; Tensor Parallel and Pipeline Parallel for models large enough that
even FSDP's sharding isn't sufficient on your hardware, often combined
together (as in Megatron-LM-style training).
</details>

**3. How do you safely take a trained model to production?**
<details><summary>Answer</summary>
Register the checkpoint with its evaluation metrics attached, validate it
in staging, run it as shadow traffic to compare against the current
production model without affecting users, then roll it out to a small
canary percentage of real traffic with monitoring and an automatic
rollback trigger, before finally moving to 100% production traffic.
</details>


In [11]:

section("Summary")
console.print(Panel(
    "[bold]What we did:[/bold]\n"
    "  1. Built a REAL, runnable evaluation harness (scoring, thresholds, pass/fail)\n"
    "  2. Built a REAL toy model registry (file-based, version + metrics + stage tracking)\n"
    "  3. (On GPU) showed where to plug in the real Topic 2 LoRA model for evaluation\n"
    "  4. Computed REAL latency percentiles (TTFT/TPOT/p50/p95/p99) and tokens/sec from data\n"
    "  5. Referenced vLLM serving for production deployment\n"
    "  6. Ran a REAL canary-rollout simulation with pass/fail and automatic rollback\n\n"
    "[bold]Series complete![/bold] You've now gone from 'how do I teach a model a task' (Topic 1) "
    "all the way to 'how do I make fine-tuning a reliable, production-grade AI engineering "
    "practice' (Topic 6) -- on one consistent running example throughout.",
    title="Topic 6 complete -- series complete", style="bold green", box=box.DOUBLE,
))


── Summary ─────────────────────────────────────────────────────

╔══════════════════════════════════════ Topic 6 complete -- series complete ══════════════════════════════════════╗
║ What we did:                                                                                                    ║
║   1. Built a REAL, runnable evaluation harness (scoring, thresholds, pass/fail)                                 ║
║   2. Built a REAL toy model registry (file-based, version + metrics + stage tracking)                           ║
║   3. (On GPU) showed where to plug in the real Topic 2 LoRA model for evaluation                                ║
║   4. Computed REAL latency percentiles (TTFT/TPOT/p50/p95/p99) and tokens/sec from data                         ║
║   5. Referenced vLLM serving for production deployment                                                          ║
║   6. Ran a REAL canary-rollout simulation with pass/fail and automatic rollback                                 ║
║                                                                                                                 ║
║ Series complete! You've now gone from 'how do I teach a model a task' (Topic 1) all the way to 'how do I make   ║
║ fine-tuning a reliable, production-grade AI engineering practice' (Topic 6) -- on one consistent running        ║
║ example throughout.                                                                                             ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝